# Module 1 – Data: loading and inspection

Load the reviews (`recommendations.csv`) and the two game files (Kozyriev `games.csv` and FronkonGames `games_1.csv`), then look at: size, dtypes, missing values, duplicates.

**Nothing** is dropped or imputed at this step.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from config import *

NROWS = 5_000_000  # development sample (the full file has 41M rows)

## 1. Reviews

In [2]:
reviews = pd.read_csv(
    RAW_DIR / "recommendations.csv",
    usecols=["app_id", "date", "is_recommended", "hours", "helpful"],
    dtype={"app_id": "int32", "is_recommended": "bool", "hours": "float32", "helpful": "int32"},
    nrows=NROWS,
)
reviews["date"] = pd.to_datetime(reviews["date"], format="%Y-%m-%d", errors="coerce")

print("Shape:", reviews.shape)
reviews.dtypes

Shape: (5000000, 5)


app_id                     int32
helpful                    int32
date              datetime64[us]
is_recommended              bool
hours                    float32
dtype: object

In [3]:
print("% missing per column:")
print(reviews.isna().mean() * 100)
print("Duplicated rows:", reviews.duplicated().sum())

% missing per column:
app_id            0.0
helpful           0.0
date              0.0
is_recommended    0.0
hours             0.0
dtype: float64


Duplicated rows: 185657


The "duplicated rows" are not necessarily real duplicates: two players can review the same game on the same day with the same hours played. So they are not dropped.

## 2. Kozyriev games (`games.csv`)

In [4]:
games = pd.read_csv(RAW_DIR / "games.csv")
games["date_release"] = pd.to_datetime(games["date_release"], format="%Y-%m-%d", errors="coerce")

print("Shape:", games.shape)
games.dtypes

Shape: (50872, 13)


app_id                     int64
title                        str
date_release      datetime64[us]
win                         bool
mac                         bool
linux                       bool
rating                       str
positive_ratio             int64
user_reviews               int64
price_final              float64
price_original           float64
discount                 float64
steam_deck                  bool
dtype: object

In [5]:
print("% missing per column:")
print(games.isna().mean() * 100)
print("Duplicated app_id:", games["app_id"].duplicated().sum())

% missing per column:
app_id            0.0
title             0.0
date_release      0.0
win               0.0
mac               0.0
linux             0.0
rating            0.0
positive_ratio    0.0
user_reviews      0.0
price_final       0.0
price_original    0.0
discount          0.0
steam_deck        0.0
dtype: float64
Duplicated app_id: 0


## 3. FronkonGames games (`games_1.csv`)

The file header is shifted: `Discount` and `DLC count` are merged into a single column, `DiscountDLC count`, although every row does contain both values. We fix the column names before reading the file, otherwise every column after `Price` is shifted by one.

Only the useful columns are kept (`usecols`), because the file is large (400 MB).

In [6]:
path = RAW_DIR / "games_1.csv"

# Header fix: "DiscountDLC count" -> "Discount", "DLC count"
cols = pd.read_csv(path, nrows=0).columns.tolist()
i = cols.index("DiscountDLC count")
cols[i:i + 1] = ["Discount", "DLC count"]

fronkon = pd.read_csv(
    path,
    names=cols,
    skiprows=1,  # skip the original header, replaced by cols
    usecols=["AppID", "Name", "Release date", "Required age", "Price", "DLC count",
             "About the game", "Supported languages", "Windows", "Mac", "Linux",
             "Categories", "Genres", "Tags"],
)
fronkon["Release date"] = pd.to_datetime(fronkon["Release date"], format="%b %d, %Y", errors="coerce")

print("Shape:", fronkon.shape)
fronkon.head(3)  # check that the values are in the right columns

Shape: (125855, 14)


,AppID,Name,Release date,Required age,Price,DLC count,About the game,Supported languages,Windows,Mac,Linux,Categories,Genres,Tags
0,2539430,Black Dragon Mage Playtest,2023-08-01,0,0.00,0,NaN,[],True,False,False,NaN,NaN,NaN
1,496350,Supipara - Chapter 1 Spring Has Come!,2016-07-29,0,5.24,0,"Springtime, April: when the cherry trees come ...",['English'],True,False,False,"Single-player,Steam Trading Cards,Steam Cloud,...",Adventure,"Adventure,Visual Novel,Anime,Cute"
2,1034400,Mystery Solitaire The Black Raven,2019-05-06,0,4.99,0,"Immerse yourself in the most beloved, mystical...","['English', 'French', 'German', 'Russian']",True,True,False,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec..."


Dates use the US format (`Aug 1, 2023`). A date that does not follow this format (e.g. `Aug 2023` or `Coming soon`) becomes `NaT`: it shows up in the missing values of `Release date` below, and is not dropped.

In [7]:
print("Missing values per column:")
print(fronkon.isna().sum())
print("Duplicated AppID:", fronkon["AppID"].duplicated().sum())

Missing values per column:
AppID                      0
Name                       1
Release date               0
Required age               0
Price                      0
DLC count                  0
About the game          8462
Supported languages        0
Windows                    0
Mac                        0
Linux                      0
Categories              8966
Genres                  8423
Tags                   42502
dtype: int64
Duplicated AppID: 0


## 4. `reviews_clean`: reviews of the game subset

We draw a subset of games **by `app_id`** (games released between 2015 and 2021), then keep **all** their reviews. Review rows are never sampled: that would distort the number of reviews per game, hence the target.

The release date comes from Kozyriev (`date_release`), which is our base table.

In [8]:
N_GAMES = 10_000  # number of games drawn at random (None = all 2015-2021 games)

subset = games[(games["date_release"] >= "2015-01-01") & (games["date_release"] <= "2021-12-31")]
if N_GAMES is not None:
    subset = subset.sample(N_GAMES, random_state=SEED)

release = subset.set_index("app_id")["date_release"]  # app_id -> release date
print("Games selected:", len(release))

Games selected: 10000


In [9]:
# The full file does not fit in memory: read it in chunks
# and keep only the reviews of the selected games.
chunks = []
for chunk in pd.read_csv(
    RAW_DIR / "recommendations.csv",
    usecols=["app_id", "date", "is_recommended", "hours", "helpful"],
    dtype={"app_id": "int32", "is_recommended": "bool", "hours": "float32", "helpful": "int32"},
    chunksize=2_000_000,
):
    chunks.append(chunk[chunk["app_id"].isin(release.index)])

reviews_clean = pd.concat(chunks, ignore_index=True)
reviews_clean["date"] = pd.to_datetime(reviews_clean["date"], format="%Y-%m-%d", errors="coerce")
assert reviews_clean["date"].notna().all(), "some review dates are unreadable"

print("Reviews of the selected games:", len(reviews_clean))
print("Games with at least 1 review:", reviews_clean["app_id"].nunique())

Reviews of the selected games: 7304209
Games with at least 1 review: 7906


### `days`: number of days between the game release and the review

A review with `days < 0` was posted **before** the official release date, usually during early access. We count them, then exclude them: the target and the early signals are measured from the release.

In [10]:
reviews_clean["days"] = (reviews_clean["date"] - reviews_clean["app_id"].map(release)).dt.days

n_before = (reviews_clean["days"] < 0).sum()
print(f"Reviews before release (days < 0): {n_before} ({n_before / len(reviews_clean):.2%}) -> excluded")
reviews_clean = reviews_clean[reviews_clean["days"] >= 0]

print("Reviews kept:", len(reviews_clean))
reviews_clean.head()

Reviews before release (days < 0): 470961 (6.45%) -> excluded
Reviews kept: 6833248


,app_id,helpful,date,is_recommended,hours,days
0,306130,0,2021-10-10,True,8.600000,1602
1,255710,0,2021-05-21,True,18.700001,2264
2,306130,0,2021-05-12,True,61.099998,1451
3,635260,0,2022-01-30,True,177.000000,1535
4,518790,0,2022-12-31,True,10.000000,2144


In [11]:
reviews_clean = reviews_clean[["app_id", "date", "days", "is_recommended", "hours", "helpful"]]
reviews_clean.to_parquet(PROC_DIR / "reviews_clean.parquet", index=False)
print("Saved:", PROC_DIR / "reviews_clean.parquet")

Saved: C:\Users\famil\Documents\ESILV\A4\S1\Machine Learning\SteamGameIA\data\processed\reviews_clean.parquet


## 5. `games_clean`: merging Kozyriev + FronkonGames

The game subset (Kozyriev base table) is merged with FronkonGames on `app_id`:
- **`left` join**: every sampled game is kept, even those missing from FronkonGames (their FronkonGames columns stay empty);
- **`validate="one_to_one"`**: pandas raises an error if an `app_id` is duplicated on either side, which would duplicate rows;
- **`indicator=True`**: the `_merge` column tells, for each game, whether it was found in FronkonGames.

`games_clean` contains **all** the sampled games, including those with no review at all. Module 2 needs them to give them 0 reviews.

In [12]:
fronkon = fronkon.rename(columns={"AppID": "app_id"})
merged = subset.merge(fronkon, on="app_id", how="left", validate="one_to_one", indicator=True)
assert merged["app_id"].is_unique

print("Games in the subset:", len(subset))
print("Games after merge:  ", len(merged))
print(merged["_merge"].value_counts())  # both = found in FronkonGames, left_only = missing

# Comparison of the two release dates
diff = (merged["Release date"] - merged["date_release"]).dt.days.abs()
print("Release dates differing by more than 30 days:", (diff > 30).sum())

Games in the subset: 10000
Games after merge:   10000
_merge
both          7797
left_only     2203
right_only       0
Name: count, dtype: int64
Release dates differing by more than 30 days: 143


### Columns kept

- **Kept:** id, title, release date (Kozyriev), base price `price_original`, platforms, and the FronkonGames columns (required age, DLC count, description, languages, categories, genres, tags).
- **Excluded, because measured after release (data leakage):** `rating`, `positive_ratio`, `user_reviews`, `price_final`, `discount`, `steam_deck`, and the FronkonGames `Price` (price at collection time, not at launch).
- **Excluded, because duplicated:** `Name` (= `title`), `Windows` / `Mac` / `Linux` (= `win` / `mac` / `linux`), `Release date` (we keep the Kozyriev date).

⚠️ `dlc_count` and `tags` may have changed after release: this is a leakage risk, to be discussed in Module 3.

In [13]:
games_clean = merged.rename(columns={
    "date_release": "release_date",
    "Required age": "required_age",
    "DLC count": "dlc_count",
    "About the game": "description",
    "Supported languages": "languages",
    "Categories": "categories",
    "Genres": "genres",
    "Tags": "tags",
})[["app_id", "title", "release_date", "price_original", "win", "mac", "linux",
    "required_age", "dlc_count", "description", "languages", "categories", "genres", "tags"]]

games_clean.to_parquet(PROC_DIR / "games_clean.parquet", index=False)
print("Saved:", PROC_DIR / "games_clean.parquet", games_clean.shape)

Saved: C:\Users\famil\Documents\ESILV\A4\S1\Machine Learning\SteamGameIA\data\processed\games_clean.parquet (10000, 14)
